In [1]:
# install piccard and piccard2
import sys
import os

sys.path.append(os.path.abspath('../src/piccard'))

import piccard2 as pc2

# install other dependencies, pip install first if needed
from tscluster.tsplot import tsplot
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

C:\Users\ecorb\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
canada_lowincome_2021 = gpd.read_file('piccard2_testing_data/canada_lowincome_2021.geojson')
canada_lowincome_2016 = gpd.read_file('piccard2_testing_data/canada_lowincome_2016.geojson')
canada_lowincome_2011 = gpd.read_file('piccard2_testing_data/canada_lowincome_2011.geojson')
canada_lowincome_2006 = gpd.read_file('piccard2_testing_data/canada_lowincome_2006.geojson')

In [3]:
big_census_dfs = [canada_lowincome_2006, canada_lowincome_2011, canada_lowincome_2016, canada_lowincome_2021]
big_years = ['2006', '2011', '2016', '2021']

t = pc2.create_network(big_census_dfs, big_years, 'GeoUID', 0.05)

Preprocessing complete
CT_CONTAINMENT: total rows = 22496 | parallel = True
ct_containment done
All nodes found
All attributes found
Graph created


In [4]:
network_table = pc2.create_network_table(big_census_dfs, big_years, 'GeoUID')

Preprocessing complete
CT_CONTAINMENT: total rows = 22496 | parallel = True
All nodes found
All possible paths through the graph found
All attributes found
Table created


In [ ]:
households_data_2021 = gpd.read_file("https://raw.githubusercontent.com/ecorbin567/piccard2/refs/heads/main/docs/piccard2_testing_data/households_data_2021.geojson")
households_data_2016 = gpd.read_file("https://raw.githubusercontent.com/ecorbin567/piccard2/refs/heads/main/docs/piccard2_testing_data/households_data_2016.geojson")
households_data_2011 = gpd.read_file("https://raw.githubusercontent.com/ecorbin567/piccard2/refs/heads/main/docs/piccard2_testing_data/households_data_2011.geojson")
households_data_2006 = gpd.read_file("https://raw.githubusercontent.com/ecorbin567/piccard2/refs/heads/main/docs/piccard2_testing_data/households_data_2006.geojson")

households_data_2021.rename(columns={'v_CA21_434: Occupied private dwellings by structural type of dwelling data': 'occupied_private_dwellings',
                                     'v_CA21_435: Single-detached house': 'single_detached_house',
                                     'v_CA21_440: Apartment in a building that has five or more storeys': 'apt_five_or_more'}, inplace=True)
households_data_2016.rename(columns={'v_CA16_408: Occupied private dwellings by structural type of dwelling data': 'occupied_private_dwellings',
                                     'v_CA16_409: Single-detached house': 'single_detached_house',
                                     'v_CA16_410: Apartment in a building that has five or more storeys': 'apt_five_or_more'}, inplace=True)
households_data_2011.rename(columns={'v_CA11F_199: Total number of occupied private dwellings by structural type of dwelling': 'occupied_private_dwellings',
                                     'v_CA11F_200: Single-detached house': 'single_detached_house',
                                     'v_CA11F_201: Apartment, building that has five or more storeys': 'apt_five_or_more',}, inplace=True)
households_data_2006.rename(columns={'v_CA06_119: Total number of occupied private dwellings by structural type of dwelling - data': 'occupied_private_dwellings',
                                     'v_CA06_120: Single-detached house': 'single_detached_house',
                                     'v_CA06_124: Apartment, building that has five or more storeys': 'apt_five_or_more',}, inplace=True)

In [ ]:
census_dfs = [households_data_2006, households_data_2011, households_data_2016, households_data_2021]
years = ['2006', '2011', '2016', '2021']

network_table = pc2.create_network_table(census_dfs, years, 'GeoUID')
G = pc2.create_network(census_dfs, years, 'GeoUID', 0.05)

In [ ]:
network_table[14:17]

In [ ]:
arr, label_dict = pc2.clustering_prep(network_table, 'name', [
    'occupied_private_dwellings_2006', 'single_detached_house_2006', 'apt_five_or_more_2006',
    'occupied_private_dwellings_2011', 'single_detached_house_2011', 'apt_five_or_more_2011',
    'occupied_private_dwellings_2016', 'single_detached_house_2016', 'apt_five_or_more_2016',
    'occupied_private_dwellings_2021', 'single_detached_house_2021', 'apt_five_or_more_2021'])

In [ ]:
tsc = pc2.cluster(network_table, G, 'GeoUID', 4, arr=arr, label_dict=label_dict)

In [ ]:
pc2.plot_subnetwork(network_table, G, paths_to_show=[14, 15, 16])

In [ ]:
pc2.plot_subnetwork(network_table, G)

In [ ]:
pc2.plot_num_areas(network_table)

In [ ]:
# %pip install nbformat
new_label_dict = {'T': label_dict['T'], 'N': label_dict['N'], 'F': ['Occupied Private Dwellings', 'Single Detached Houses', 'Apartments in a building with five or more storeys']}

figs = pc2.plot_clusters_scatter(
    tsc=tsc,
    network_table=network_table,
    arr=arr,
    label_dict=new_label_dict,
    clusters_to_show = [3],
    dynamic_paths_only = False,
)

for fig in figs:
    fig.show()

In [ ]:
pc2.plot_clusters_parallelcats(
    network_table=network_table, 
    years=['2006', '2011', '2016', '2021'], 
    cluster_colours={0:'blue', 1:'red'},
    cluster_labels=['Cluster Zero', 'Cluster One', 'Cluster Two', 'Cluster Three'])

In [ ]:
pc2.plot_clusters_area(
    network_table=network_table,
    years=["2006", "2011", "2016", "2021"],
    cluster_colours={0:'blue', 1:'red'},
    cluster_labels=['Cluster Zero', 'Cluster One', 'Cluster Two', 'Cluster Three'],
    figsize=(1000, 600),
    stacked=True
)

In [ ]:
pc2.plot_clusters_map(
    year='2021',
    geofile_path=f"piccard2_testing_data/households_data_2021.geojson",
    network_table=network_table,
    label_dict=label_dict,
    cluster_colours={0:'blue', 1:'red', 3:'yellow'},
    cluster_labels=['Cluster Zero', 'Cluster One', 'Cluster Two', 'Cluster Three'],
)